3 silver.douyin_aweme_media

Each row = 1 file media. (from manifest_media)

Field: account_id
aweme_id
media_type
media_index
s3_url
s3_key
bytes
content_type
status
error


In [0]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window


bronze_table = "de_e2e.bronze.douyin_media_manifest_raw"
silver_table = "de_e2e.silver.douyin_aweme_media"

bucket = "de-e2e-413612133697-ap-southeast-1-an"
silver_path = f"s3://{bucket}/lakehouse/silver/douyin/aweme_media_delta/"

In [0]:
manifest_df = spark.read.table(bronze_table)

media_df = (
    manifest_df
    .withColumn("item", F.explode_outer("raw_struct.items"))
    .select(
        F.col("niche"),
        F.col("account_id"),
        F.col("source_file"),
        F.col("generated_at").alias("manifest_generated_at"),
        F.col("bronze_ingested_at"),

        F.col("item.status").alias("status"),
        F.col("item.aweme_id").cast("string").alias("aweme_id"),
        F.col("item.media_type").alias("media_type"),
        F.col("item.index").cast("int").alias("media_index"),
        F.col("item.s3_key").alias("s3_key"),
        F.col("item.s3_url").alias("s3_url"),
        F.col("item.bytes").cast("long").alias("bytes"),
        F.col("item.content_type").alias("content_type"),
        F.col("item.source_url").alias("source_url"),

        F.current_timestamp().alias("silver_updated_at"),
    )
    .where(F.col("media_type").isNotNull())
    .where(F.col("media_index").isNotNull())
)

window_spec = Window.partitionBy(
    "account_id",
    "aweme_id",
    "media_type",
    "media_index",
).orderBy(
    F.col("manifest_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
    F.col("source_file").desc_nulls_last(),
)

media_clean_df = (
    media_df
    .withColumn("rn", F.row_number().over(window_spec))
    .where(F.col("rn") == 1)
    .drop("rn")
)

display(
    media_clean_df.select(
        "account_id",
        "aweme_id",
        "media_type",
        "media_index",
        "status",
        "s3_url",
        "bytes",
        "content_type",
    ).orderBy("account_id", "aweme_id", "media_type", "media_index").limit(100)
)

In [0]:
from delta.tables import DeltaTable
if DeltaTable.isDeltaTable(spark, silver_path):
    target = DeltaTable.forPath(spark, silver_path)
    (
        target.alias("t")
        .merge(
            media_clean_df.alias("s"),
            """
            t.account_id = s.account_id
            AND t.aweme_id = s.aweme_id
            AND t.media_type = s.media_type
            AND t.media_index = s.media_index
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        media_clean_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    LOCATION '{silver_path}'
    """
)

In [0]:
result_df = spark.table(silver_table)

print(f"Silver media rows: {result_df.count()}")

display(
    result_df.groupBy("media_type", "status")
    .agg(
        F.count("*").alias("file_count"),
        F.countDistinct("aweme_id").alias("aweme_count"),
        F.sum("bytes").alias("total_bytes"),
    )
    .orderBy("media_type", "status")
)